# JetBrains Onboarding A/B Test — Activation Analysis

**Dataset:** `License_activation_ab_test_dataset_3ver.csv`  
**Goal:** Understand what drives user activation across experiment groups A and B  
**Key question:** Why does Group B activate at a higher rate than Group A?

---

### Columns
| Column | Description |
|---|---|
| `user_id` | Unique user identifier |
| `experiment_group` | A or B |
| `activated` | 1 = activated, 0 = not activated |
| `time_to_first_run_min` | Minutes from install to first IDE run |
| `used_autocomplete_day1` | Used autocomplete feature on day 1 (0/1) |
| `used_refactoring_day1` | Used refactoring feature on day 1 (0/1) |

## 0 · Setup & Data Load

In [ ]:
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Dark theme
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2a2d3e',
    'grid.color':       '#2a2d3e',
    'text.color':       '#e2e4ef',
    'axes.labelcolor':  '#e2e4ef',
    'xtick.color':      '#7a7d9a',
    'ytick.color':      '#7a7d9a',
    'axes.titlecolor':  '#e2e4ef',
    'legend.facecolor': '#1a1d27',
    'legend.edgecolor': '#2a2d3e',
    'savefig.facecolor':'#0f1117',
})

PURPLE = '#7c6af7'
TEAL   = '#4ecdc4'
GREEN  = '#4ade80'
RED    = '#f87171'
YELLOW = '#ffe66d'
ORANGE = '#f97b6b'

# Direct path to your CSV file
CSV = r'C:\Users\Артём\Desktop\Claude_work\License_activation_ab_test_dataset_3ver.csv'

df = pd.read_csv(CSV)
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head()

## 1 · Quick Data Overview

In [ ]:
print('=== Shape ===')
print(df.shape)
print('\n=== Null counts ===')
print(df.isnull().sum())
print('\n=== Descriptive stats ===')
df.describe().round(2)

In [ ]:
# Helper: Welch t-statistic (no scipy needed)
def welch_t(a, b):
    ma, mb = np.mean(a), np.mean(b)
    sa, sb = np.std(a, ddof=1), np.std(b, ddof=1)
    na, nb = len(a), len(b)
    se = math.sqrt(sa**2 / na + sb**2 / nb)
    return (ma - mb) / se

# Helper: Chi-square from contingency table
def chi2_1dof(ct):
    total = ct.values.sum()
    rs = ct.sum(axis=1).values
    cs = ct.sum(axis=0).values
    return sum(
        (ct.values[i, j] - rs[i] * cs[j] / total) ** 2 / (rs[i] * cs[j] / total)
        for i in range(ct.shape[0])
        for j in range(ct.shape[1])
    )

# Derived columns used across all plots
df['Activation Status'] = df['activated'].map({1: 'Activated', 0: 'Not Activated'})

df['time_bucket'] = pd.cut(
    df['time_to_first_run_min'],
    bins=[0, 5, 15, 30, 60, 120, np.inf],
    labels=['0-5 min', '5-15 min', '15-30 min', '30-60 min', '60-120 min', '120+ min']
)

df['Feature Combo'] = df.apply(
    lambda r: ('Auto=Y' if r['used_autocomplete_day1'] else 'Auto=N') +
              ('  Refac=Y' if r['used_refactoring_day1'] else '  Refac=N'),
    axis=1
)

print('Helper functions and derived columns ready.')

## 2 · Key Stats Summary

In [ ]:
ab = df.groupby('experiment_group')['activated'].agg(
    rate='mean', count='count', n_activated='sum').reset_index()
ab['pct'] = (ab['rate'] * 100).round(2)

ct   = pd.crosstab(df['experiment_group'], df['activated'])
chi2 = chi2_1dof(ct)
lift = (ab.loc[ab.experiment_group=='B','rate'].values[0] /
        ab.loc[ab.experiment_group=='A','rate'].values[0] - 1) * 100

act   = df[df['activated']==1]['time_to_first_run_min']
noact = df[df['activated']==0]['time_to_first_run_min']
t_act = welch_t(act.values, noact.values)

print('=' * 42)
print(' A/B Test Results')
print('=' * 42)
for _, row in ab.iterrows():
    print(f'  Group {row.experiment_group}: {row.pct:.2f}%  ({int(row.n_activated):,} / {int(row["count"]):,})')
print(f'  Lift B over A : +{lift:.2f}%')
print(f'  Chi-square    : {chi2:.2f}  (>10.83 means p < 0.001)')
print()
print('=' * 42)
print(' Time to First Run vs Activation')
print('=' * 42)
print(f'  Activated     mean={act.mean():.1f}m   median={act.median():.1f}m')
print(f'  Not Activated mean={noact.mean():.1f}m  median={noact.median():.1f}m')
print(f'  t-statistic   {t_act:.1f}  (|t| > 3 = highly significant)')

---
## Plot 1 · A/B Activation Rate Comparison

In [ ]:
ab['not_activated'] = ab['count'] - ab['n_activated']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Plot 1: A/B Test Activation Rate Comparison', fontsize=14, fontweight='bold')

# Left: activation rate bars
bars = axes[0].bar(ab['experiment_group'], ab['pct'],
                   color=[PURPLE, TEAL], width=0.5, edgecolor='white', linewidth=0.5)
for bar, row in zip(bars, ab.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{row.pct:.2f}%', ha='center', va='bottom',
                 fontweight='bold', fontsize=13, color='white')
axes[0].set_ylim(55, 74)
axes[0].set_xlabel('Experiment Group')
axes[0].set_ylabel('Activation Rate (%)')
axes[0].set_title(f'Activation Rate\nchi2={chi2:.1f}, p<0.001, Lift=+{lift:.1f}%')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

# Right: stacked counts
ab_plot = ab.set_index('experiment_group')[['n_activated', 'not_activated']]
ab_plot.plot(kind='bar', stacked=True, ax=axes[1],
             color=[GREEN, RED], edgecolor='none', width=0.5)
axes[1].set_title('Activated vs Not-Activated (stacked count)')
axes[1].set_xlabel('Experiment Group')
axes[1].set_ylabel('Number of Users')
axes[1].set_xticklabels(['Group A', 'Group B'], rotation=0)
axes[1].legend(['Activated', 'Not Activated'])
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

---
## Plot 2 · The 30-Minute Cliff

**Key finding:** Activation rate drops from ~97% for users who run within 15 minutes
to just 3.4% for users who take 30-60 minutes. It is a cliff, not a gradual slope.

In [ ]:
bucket_stats = (
    df.groupby('time_bucket', observed=True)['activated']
    .agg(rate='mean', count='count')
    .reset_index()
)
bucket_stats['pct'] = bucket_stats['rate'] * 100
bucket_colors = [GREEN, GREEN, YELLOW, RED, RED, RED]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Plot 2: The 30-Minute Cliff', fontsize=14, fontweight='bold')

# Left: activation rate
axes[0].bar(bucket_stats['time_bucket'].astype(str), bucket_stats['pct'],
            color=bucket_colors, edgecolor='white', linewidth=0.4, width=0.6)
for i, row in bucket_stats.iterrows():
    axes[0].text(i, row['pct'] + 1.5, f"{row['pct']:.1f}%",
                 ha='center', fontsize=11, fontweight='bold', color='white')
axes[0].axvline(x=2.5, color=ORANGE, linewidth=2, linestyle='--', alpha=0.8)
axes[0].text(2.65, 50, '<- 30-min cliff', color=ORANGE, fontsize=10, fontstyle='italic')
axes[0].set_ylim(0, 108)
axes[0].set_xlabel('Time to First Run Bucket')
axes[0].set_ylabel('Activation Rate (%)')
axes[0].set_title('Activation Rate per Time Bucket')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[0].tick_params(axis='x', rotation=20)

# Right: user volume
axes[1].bar(bucket_stats['time_bucket'].astype(str), bucket_stats['count'],
            color=bucket_colors, edgecolor='white', linewidth=0.4, width=0.6)
for i, row in bucket_stats.iterrows():
    axes[1].text(i, row['count'] + 60, f"{int(row['count']):,}",
                 ha='center', fontsize=10, color='white')
axes[1].axvline(x=2.5, color=ORANGE, linewidth=2, linestyle='--', alpha=0.8)
axes[1].set_xlabel('Time to First Run Bucket')
axes[1].set_ylabel('Number of Users')
axes[1].set_title('User Volume per Time Bucket')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

---
## Plot 3 · Time-to-First-Run: Activated vs Not-Activated

Activated and not-activated users have almost non-overlapping time distributions.

In [ ]:
t_stat = welch_t(act.values, noact.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Plot 3: Time to First Run — Activated vs Not-Activated', fontsize=14, fontweight='bold')

# Left: KDE
sns.kdeplot(data=df, x='time_to_first_run_min', hue='Activation Status',
            fill=True, alpha=0.35, linewidth=2,
            palette={'Activated': GREEN, 'Not Activated': RED},
            ax=axes[0], clip=(0, 80))
axes[0].axvline(act.mean(), color=GREEN, linestyle='--', linewidth=1.8,
                label=f'Activated mean ({act.mean():.1f}m)')
axes[0].axvline(noact.mean(), color=RED, linestyle='--', linewidth=1.8,
                label=f'Not-activated mean ({noact.mean():.1f}m)')
axes[0].axvline(30, color=ORANGE, linestyle=':', linewidth=2, alpha=0.8, label='30-min cliff')
axes[0].set_xlim(0, 75)
axes[0].set_xlabel('Time to First Run (minutes)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'KDE Distribution  |  t = {t_stat:.1f}, p < 0.001')
axes[0].legend(fontsize=9)

# Right: Box plot
sns.boxplot(data=df, x='Activation Status', y='time_to_first_run_min',
            palette={'Activated': GREEN, 'Not Activated': RED},
            width=0.45, linewidth=1.2, fliersize=2, ax=axes[1])
axes[1].axhline(30, color=ORANGE, linestyle=':', linewidth=2, alpha=0.8, label='30-min cliff')
axes[1].set_xlabel('')
axes[1].set_ylabel('Time to First Run (minutes)')
axes[1].set_title('Box Plot: Spread and Median')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Plot 4 · Time to First Run: Group A vs Group B

Group B reaches first-run 4.7 minutes faster at the median. The ECDF shows how much
more of Group B sits under the 30-minute cliff.

In [ ]:
a_times = df[df['experiment_group']=='A']['time_to_first_run_min']
b_times = df[df['experiment_group']=='B']['time_to_first_run_min']
t_ab    = welch_t(a_times.values, b_times.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Plot 4: Time to First Run — Group A vs Group B', fontsize=14, fontweight='bold')

# Left: histogram
bins = np.arange(0, 65, 5)
axes[0].hist(a_times.clip(upper=64), bins=bins, alpha=0.55, color=PURPLE,
             edgecolor='white', linewidth=0.3,
             label=f'Group A (median {a_times.median():.1f}m)')
axes[0].hist(b_times.clip(upper=64), bins=bins, alpha=0.55, color=TEAL,
             edgecolor='white', linewidth=0.3,
             label=f'Group B (median {b_times.median():.1f}m)')
axes[0].axvline(a_times.median(), color=PURPLE, linewidth=2, linestyle='--')
axes[0].axvline(b_times.median(), color=TEAL,   linewidth=2, linestyle='--')
axes[0].axvline(30, color=ORANGE, linewidth=1.8, linestyle=':', alpha=0.8, label='30-min cliff')
axes[0].set_xlabel('Time to First Run (minutes)')
axes[0].set_ylabel('Number of Users')
axes[0].set_title('Histogram (5-min bins)')
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Right: Empirical CDF
for times, color, label in [(a_times, PURPLE, 'Group A'), (b_times, TEAL, 'Group B')]:
    vals = np.sort(times.values)
    cdf  = np.arange(1, len(vals) + 1) / len(vals)
    axes[1].plot(vals, cdf * 100, color=color, linewidth=2.2, label=label)
axes[1].axvline(30, color=ORANGE, linewidth=2, linestyle=':', alpha=0.8, label='30-min cliff')
axes[1].axhline(50, color='#7a7d9a', linewidth=1, linestyle='--', alpha=0.5)
axes[1].set_xlim(0, 65)
axes[1].set_xlabel('Time to First Run (minutes)')
axes[1].set_ylabel('Cumulative % of Users')
axes[1].set_title(f'Empirical CDF  |  t = {t_ab:.1f}')
axes[1].legend()
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
plt.show()

---
## Plot 5 · Feature Usage — A Weaker Signal

All feature combinations produce near-identical activation rates (~64-66%).
Feature use on day 1 is not a meaningful predictor of activation.

In [ ]:
combo_stats = (
    df.groupby('Feature Combo')['activated']
    .agg(rate='mean', count='count')
    .reset_index()
    .sort_values('rate', ascending=False)
)
combo_stats['pct'] = combo_stats['rate'] * 100

feat_group = df.groupby('experiment_group')[
    ['used_autocomplete_day1', 'used_refactoring_day1']
].mean().reset_index()
feat_group.columns = ['Group', 'Autocomplete Day 1', 'Refactoring Day 1']
feat_melt = feat_group.melt(id_vars='Group', var_name='Feature', value_name='Usage Rate')
feat_melt['Usage Rate'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Plot 5: Day-1 Feature Usage — Effect on Activation', fontsize=14, fontweight='bold')

# Left: combo activation rates
bar_colors = [GREEN if v > 65 else YELLOW for v in combo_stats['pct']]
axes[0].barh(combo_stats['Feature Combo'], combo_stats['pct'],
             color=bar_colors, edgecolor='white', linewidth=0.4, height=0.5)
for idx, (_, row) in enumerate(combo_stats.iterrows()):
    axes[0].text(row['pct'] + 0.05, idx,
                 f"{row['pct']:.1f}%  (n={int(row['count']):,})",
                 va='center', fontsize=10, color='white')
axes[0].set_xlim(60, 70)
axes[0].set_xlabel('Activation Rate (%)')
axes[0].set_title('Activation by Feature Combo\n(all within ~2% — weak signal)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

# Right: feature usage by group
sns.barplot(data=feat_melt, x='Feature', y='Usage Rate', hue='Group',
            palette={'A': PURPLE, 'B': TEAL},
            edgecolor='white', linewidth=0.4, width=0.5, ax=axes[1])
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%', padding=3, fontsize=10, color='white')
axes[1].set_ylim(0, 85)
axes[1].set_xlabel('')
axes[1].set_ylabel('Usage Rate (%)')
axes[1].set_title('Feature Usage Rates by Group\n(Group B uses both features more)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].legend(title='Group')

plt.tight_layout()
plt.show()

---
## Summary of Findings

| # | Finding | Strength |
|---|---|---|
| 1 | **Group B activates 12.8% more** than Group A (68.4% vs 60.7%) | chi2=99.4, p<0.001 |
| 2 | **Time to first run is the #1 predictor** — users under 15 min activate at 97%; 30+ min drops to 3.4% | t=-123.6, p<0.001 |
| 3 | **Group B reaches first run 4.7 min faster** (median 17.7m vs 22.4m) — this explains B's lift | t=18.5, p<0.001 |
| 4 | **Feature usage barely predicts activation** — all combos within a 2% band | No signal |

### Recommendations
1. Make 'first run under 15 minutes' your North Star metric.
2. Ship Group B's onboarding broadly — the evidence is conclusive.
3. Investigate what Group B does differently in the first 17 minutes.
4. Do not delay first run to showcase features — feature exposure on day 1 does not move activation.
5. Focus friction reduction on the ~25% of users taking 30+ minutes (only 3.4% activate).